# REMDB v4 Data Processing for TARE Model

**Purpose**: Transform the raw REMDB v4 Excel file into a clean, flat format suitable for Python-based capital cost calculations.

**Input**: `REMDB_2024_12_23.xlsx` (raw NREL REMDB v4 file)  
**Output**: `remdb_v4_tare_costs.xlsx` (clean, flat file with TARE-relevant components)

**Key Transformations**:
1. Flatten multi-level column headers into Python-friendly names
2. Filter to TARE-relevant components (heating, cooling, water heating, etc.)
3. Add category labels and unique row identifiers
4. Determine installation method (multiplier vs. adder) for each component

**Data Year**: All costs are in 2023$ — no CPI adjustment needed.

In [1]:
# Imports and Configuration
# Load required libraries and set display options for data exploration.
from config import PROJECT_ROOT

import os
import pandas as pd
import numpy as np
import re
from pathlib import Path

# Display settings for better DataFrame viewing
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.width', 200)

# =============================================================================
# FILE PATHS - Update these to match your environment
# =============================================================================
REMBD_RAW_PATH = os.path.join(PROJECT_ROOT, 'cmu_tare_model', 'data', 'retrofit_costs', 'REMDB_2024_12_23.xlsx')  # Raw REMDB v4 file path

print(f"Input file path: {REMBD_RAW_PATH}")
# print(f"Output file path: {REMBD_PROCESSED_PATH}")

Project root directory: c:\users\jorda\desktop\projects\cmu-tare-model
Input file path: c:\users\jorda\desktop\projects\cmu-tare-model\cmu_tare_model\data\retrofit_costs\REMDB_2024_12_23.xlsx


In [2]:
df_remdb_raw = pd.read_excel(REMBD_RAW_PATH, sheet_name='Machine Read') 
df_remdb_raw

,Component & Class,Unnamed: 1,Unnamed: 2,Unnamed: 3,"Retail Price Regression, Performance metric 1",Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,"Retail Price Regression, Performance metric 2",Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,"Retail Price Regression, Intercepts",Unnamed: 19,Unnamed: 20,Installation Multiplier,Unnamed: 22,Installation Adder,Unnamed: 24,Unnamed: 25,Additional Data,Unnamed: 27,Unnamed: 28,Unnamed: 29,Unnamed: 30,Unnamed: 31
0,Name,Class,Class - additional description,Output Units,Coefficient-Low,Coefficient-Mid,Coefficient-High,Metric,Unit,Lower Bound,Upper Bound,Coefficient-Low,Coefficient-Mid,Coefficient-High,Metric,Unit,Lower Bound,Upper Bound,Int-Low,Int-Mid,Int-High,New Construction,Retrofit,New Construction,Retrofit,NaN,Lifetime,Cost Variation Considerations,Data Sources,Qualitative Rank,Notes,NaN
1,Air Conditioner,Centrally ducted,NaN,2023$,196.966095,328.276825,459.587555,Cooling Capacity,Tons,1.5,5,356.844,594.74,832.636,SEER1,Unitless,13,24,-3533.36167,-5888.936117,-8244.510564,1.5,1.5,-327.407407,0,NaN,22.333333,"Prevailing local wages, Drive time, Local code...","1, 2, 3, 4, 14","Low SS, Medium R2, High Source Diversity",Uses SEER1 coefficient for ASHPs from [14] to ...,NaN
2,Air Conditioner,Room AC (window or through-wall),NaN,2023$,350.16,364.332,311.64,Capacity,Tons,0.416667,2.333333,2.082501,6.069364,25.478424,CEER,Unitless,9.4,15,0.194766,13.27746,2.001877,1,1,199.557677,352.607677,NaN,10,"Prevailing local wages, Drive time, Local code...","1, 4","Medium SS, High R2, Medium Source Diversity",CEER metric does not scale well with predicted...,NaN
3,Air Sealing,<40% Reduction,NaN,2023$/sqft,0.48045,0.894236,1.333659,Leakage Reduction,Percent,0.03,0.4,0,0,0,NaN,NaN,0,0,0.140081,0.227373,0.457668,1,1,0,0,NaN,999,"Prevailing local wages, drive time, local code...",6,"Data inflated to 2023. High SS, Low R2",Regression based on total installed cost,NaN
4,Air Sealing,>40% Reduction,NaN,2023$/sqft,0.107294,0.867979,0.785978,Leakage Reduction,Percent,0.4,0.92,0,0,0,NaN,NaN,0,0,0.252077,0.297935,0.874814,1,1,0,0,NaN,999,"Prevailing local wages, drive time, local code...",6,"Data inflated to 2023. Medium SS, Low R2",Regression based on total installed cost,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
129,Wood/Steel Stud,Loose Fill,NaN,2023$/sqft,0.003619,0.00704,0.001017,R-value,ft2·°F·h/BTU,2.78,38,0,0,0,NaN,NaN,0,0,0.156253,0.278882,0.955934,1.730579,1.809917,0,0,NaN,999,"Prevailing local wages, Drive time, Local code...","1, 2, 4","Low SS, Low R2, High Source Diversity",NaN,NaN
130,Wood/Steel Stud,Rigid Foam (ISO),NaN,2023$/sqft,0.026871,0.101739,0.144589,R-value,ft2·°F·h/BTU,2.7,31,0,0,0,NaN,NaN,0,0,0.536513,0.490991,0.670235,1,1,0.40875,0.47875,NaN,999,"Prevailing local wages, Drive time, Local code...","1, 2, 4","Low SS, High R2, High Source Diversity",NaN,NaN
131,Wood/Steel Stud,Rigid Foam (XPS),NaN,2023$/sqft,0.123717,0.1755,0.2285,R-value,ft2·°F·h/BTU,1,20,0,0,0,NaN,NaN,0,0,0.065725,0.101252,2.7075,1,1,0.455,0.525,NaN,999,"Prevailing local wages, Drive time, Local code...","1, 2, 4","Medium SS, Medium R2, High Source Diversity",NaN,NaN
132,Wood/Steel Stud,Spray Foam (Closed Cell),NaN,2023$/sqft,0.209286,0.301055,0.618535,R-value,ft2·°F·h/BTU,5.5,42,0,0,0,NaN,NaN,0,0,0.000004,0.031017,-0.003062,1.13668,1.284942,0,0,NaN,999,"Prevailing local wages, Drive time, Local code...","1, 2, 4","High SS, Medium R2, High Source Diversity",NaN,NaN


In [3]:
"""
Load Raw Data
---------------------
Read the REMDB v4 Excel file with multi-level headers.
"""
# Read the main data sheet with multi-level headers
df_remdb_raw = pd.read_excel(REMBD_RAW_PATH, sheet_name='Machine Read', header=[0, 1])
print(df_remdb_raw)

print(f"\nRaw data shape: {df_remdb_raw.shape[0]} rows × {df_remdb_raw.shape[1]} columns")
print(f"Column levels: {df_remdb_raw.columns.nlevels}")

# Show first few raw column names to verify structure
print("\nFirst 5 columns (multi-level):")
for i, col in enumerate(df_remdb_raw.columns[:5]):
    print(f"  {i}: Level0='{col[0]}' | Level1='{col[1]}'")

        Component & Class                                                                                                   Retail Price Regression, Performance metric 1                  \
                     Name                             Class                     Class - additional description Output Units                               Coefficient-Low Coefficient-Mid   
0         Air Conditioner                  Centrally ducted                                                NaN        2023$                                    196.966095      328.276825   
1         Air Conditioner  Room AC (window or through-wall)                                                NaN        2023$                                    350.160000      364.332000   
2             Air Sealing                    <40% Reduction                                                NaN   2023$/sqft                                      0.480450        0.894236   
3             Air Sealing                    >40% Reduc

In [4]:
"""
Define Column Mapping
-----------------------------
The raw REMDB v4 file has multi-level column headers (2 rows).
This mapping converts those to flat, Python-friendly column names.

Format: (Level0, Level1) -> new_name
"""

COLUMN_MAPPING = {
    # =========================================================================
    # COMPONENT IDENTIFICATION
    # These columns identify what equipment/component each row represents
    # =========================================================================
    ('Component & Class', 'Name'): 'component',
    ('Component & Class', 'Class'): 'class',
    ('Component & Class', 'Class - additional description'): 'class_description',
    ('Component & Class', 'Output Units'): 'output_units',
    
    # =========================================================================
    # PERFORMANCE METRIC 1 (Primary regression variable)
    # Most components have at least one performance metric that drives cost
    # Examples: Cooling Capacity (tons), UEF, R-value
    # =========================================================================
    ('Retail Price Regression, Performance metric 1', 'Coefficient-Low'): 'pm1_coef_low',
    ('Retail Price Regression, Performance metric 1', 'Coefficient-Mid'): 'pm1_coef_mid',
    ('Retail Price Regression, Performance metric 1', 'Coefficient-High'): 'pm1_coef_high',
    ('Retail Price Regression, Performance metric 1', 'Metric'): 'pm1_metric',
    ('Retail Price Regression, Performance metric 1', 'Unit'): 'pm1_unit',
    ('Retail Price Regression, Performance metric 1', 'Lower Bound'): 'pm1_lower_bound',
    ('Retail Price Regression, Performance metric 1', 'Upper Bound'): 'pm1_upper_bound',
    
    # =========================================================================
    # PERFORMANCE METRIC 2 (Secondary regression variable, not always used)
    # Some components have a second metric (e.g., SEER for heat pumps)
    # These columns will be NaN for components with only one metric
    # =========================================================================
    ('Retail Price Regression, Performance metric 2', 'Coefficient-Low'): 'pm2_coef_low',
    ('Retail Price Regression, Performance metric 2', 'Coefficient-Mid'): 'pm2_coef_mid',
    ('Retail Price Regression, Performance metric 2', 'Coefficient-High'): 'pm2_coef_high',
    ('Retail Price Regression, Performance metric 2', 'Metric'): 'pm2_metric',
    ('Retail Price Regression, Performance metric 2', 'Unit'): 'pm2_unit',
    ('Retail Price Regression, Performance metric 2', 'Lower Bound'): 'pm2_lower_bound',
    ('Retail Price Regression, Performance metric 2', 'Upper Bound'): 'pm2_upper_bound',
    
    # =========================================================================
    # INTERCEPTS (Constant term in regression equation)
    # Material_Price = (pm1_coef × Metric1) + (pm2_coef × Metric2) + Intercept
    # =========================================================================
    ('Retail Price Regression, Intercepts', 'Int-Low'): 'intercept_low',
    ('Retail Price Regression, Intercepts', 'Int-Mid'): 'intercept_mid',
    ('Retail Price Regression, Intercepts', 'Int-High'): 'intercept_high',
    
    # =========================================================================
    # INSTALLATION COSTS
    # Two methods exist:
    #   - Multiplier: Installed_Cost = Material_Price × Multiplier
    #   - Adder: Installed_Cost = Material_Price + Adder
    # WE USE THE RETOFIT OPTION NOT THE NEW CONSTRUCTION OPTION
    # =========================================================================
    ('Installation Multiplier', 'New Construction'): 'install_mult_new',
    ('Installation Multiplier', 'Retrofit'): 'multiplier_retrofit',
    ('Installation Adder', 'New Construction'): 'install_add_new',
    ('Installation Adder', 'Retrofit'): 'adder_retrofit',
    ('Installation Adder', 'Retrofit.1'): 'install_add_retrofit_alt',  # Duplicate in source
    
    # =========================================================================
    # ADDITIONAL METADATA
    # =========================================================================
    ('Additional Data', 'Lifetime'): 'lifetime_years',
    ('Additional Data', 'Cost Variation Considerations'): 'cost_variation_notes',
    ('Additional Data', 'Data Sources'): 'data_sources',
    ('Additional Data', 'Qualitative Rank'): 'data_quality_rank',
    ('Additional Data', 'Notes'): 'notes',
    ('Additional Data', 'Notes.1'): 'notes_additional',
}

In [5]:
# Flatten Column Names
# Convert multi-level headers to flat, Python-friendly names.

def flatten_columns(
        df: pd.DataFrame,
        column_mapping: dict) -> pd.DataFrame:
    """
    Convert multi-level column headers to flat, single-level names.
    
    Args:
        df: DataFrame with multi-level columns (from reading Excel with header=[0,1])
        column_mapping: Dictionary mapping (Level0, Level1) tuples to new column names
        
    Returns:
        DataFrame with flattened column names
    """
    new_columns = []
    for col in df.columns:
        if col in column_mapping:
            new_columns.append(column_mapping[col])
        else:
            # Fallback for unmapped columns: create name from both levels
            fallback = f"{col[0]}_{col[1]}".lower().replace(' ', '_')
            new_columns.append(fallback)
            print(f"  Warning: Unmapped column {col} → {fallback}")
    
    df.columns = new_columns
    return df

df_remdb_flat = df_remdb_raw.copy()
df_remdb_flat = flatten_columns(df_remdb_raw, COLUMN_MAPPING)

print(f"""
DataFrame after flattening columns:
{df_remdb_flat}
""")



DataFrame after flattening columns:
                component                             class                                  class_description output_units  pm1_coef_low  pm1_coef_mid  pm1_coef_high         pm1_metric  \
0         Air Conditioner                  Centrally ducted                                                NaN        2023$    196.966095    328.276825     459.587555   Cooling Capacity   
1         Air Conditioner  Room AC (window or through-wall)                                                NaN        2023$    350.160000    364.332000     311.640000           Capacity   
2             Air Sealing                    <40% Reduction                                                NaN   2023$/sqft      0.480450      0.894236       1.333659  Leakage Reduction   
3             Air Sealing                    >40% Reduction                                                NaN   2023$/sqft      0.107294      0.867979       0.785978  Leakage Reduction   
4    Air Source He

In [6]:
# Define TARE Component Mapping
# Map REMDB v4 component names to TARE end-use categories.
TARE_COMPONENT_MAPPING = {
    # =========================================================================
    # HEATING EQUIPMENT
    # =========================================================================
    'Air Source Heat Pump': 'heating',   # Includes ducted and ductless variants
    'Furnaces': 'heating',               # Gas furnaces (fuel-agnostic in REMDB v4)
    'Boiler': 'heating',                 # Gas and oil boilers
    'Electric Baseboard': 'heating',     # Electric resistance heating
    
    # =========================================================================
    # COOLING EQUIPMENT (NEW category for TARE)
    # =========================================================================
    'Air Conditioner': 'cooling',        # Central and room AC units
    
    # =========================================================================
    # WATER HEATING
    # =========================================================================
    'Water Heater': 'waterHeating',      # All types: HPWH, electric, gas, instantaneous
    
    # =========================================================================
    # APPLIANCES
    # =========================================================================
    'Clothes Dryer': 'clothesDrying',    # Electric, heat pump, and gas dryers
    'Cooking Range': 'cooking',          # Electric, induction, and gas ranges
    
    # =========================================================================
    # ENCLOSURE/ENVELOPE UPGRADES
    # =========================================================================
    'Air Sealing': 'enclosure',              # Air leakage reduction
    'Unfinished Attic (Ceiling)': 'enclosure',  # Attic insulation
    'Duct': 'enclosure',                     # Duct insulation
    'Duct ': 'enclosure',                    # Duct sealing (note: trailing space in source data)
}

# Keep only the components relevant to the TARE model
all_components = df_remdb_flat['component'].unique()
tare_components = set(TARE_COMPONENT_MAPPING.keys())
excluded = [c for c in all_components if c not in tare_components]

print(f"""
Total components in REMDB v4: {len(all_components)}
TARE-relevant components: {len(tare_components)}
Excluded components: {len(excluded)}

Excluded components (not needed for TARE): 
{sorted(excluded)}

TARE Component Mapping:
--------------------------------------------------""")
for component, category in TARE_COMPONENT_MAPPING.items():
    print(f"  {component:<30} → {category}")


Total components in REMDB v4: 45
TARE-relevant components: 12
Excluded components: 33

Excluded components (not needed for TARE): 
['Battery', 'CMU', 'Clothes Dryer (Compact)', 'Clothes Washer', 'Crawlspace (Ceiling)', 'Crawlspace (Wall)', 'Dishwasher', 'Door', 'Double Wood Stud', 'EV Charger', 'Electric Panel', 'Exterior Finish', 'Finished Basement (Ceiling)', 'Finished Basement (Wall)', 'Finished Roof', 'Ground Source Heat Pump', 'ICF ', 'LED', 'Mechanical Ventilation', 'Pipe Insulation', 'Radiant Barrier', 'Refrigerator', 'Rim Joist', 'Slab', 'Solar PV', 'Structural Insulated Panel', 'Thermostat', 'Unfinished Attic (Roof)', 'Unfinished Basement (Ceiling)', 'Unfinished Basement (Wall)', 'Wall Sheathing', 'Window', 'Wood/Steel Stud']

TARE Component Mapping:
--------------------------------------------------
  Air Source Heat Pump           → heating
  Furnaces                       → heating
  Boiler                         → heating
  Electric Baseboard             → heating
  Air 

In [7]:
# This filters the 45+ components in REMDB v4 down to just the ones we need.
df_remdb_tare_costs = df_remdb_flat[df_remdb_flat['component'].isin(tare_components)].copy()

# Add TARE category column
df_remdb_tare_costs['tare_category'] = df_remdb_tare_costs['component'].map(TARE_COMPONENT_MAPPING)

print(f""" 
Updated TARE Retrofit Cost Dataframe using REMDB v4:
{df_remdb_tare_costs} 

Rows by TARE category:
{df_remdb_tare_costs['tare_category'].value_counts().to_string()}      
""")

 
Updated TARE Retrofit Cost Dataframe using REMDB v4:
                      component                                   class                                  class_description output_units  pm1_coef_low  pm1_coef_mid  pm1_coef_high         pm1_metric  \
0               Air Conditioner                        Centrally ducted                                                NaN        2023$    196.966095    328.276825     459.587555   Cooling Capacity   
1               Air Conditioner        Room AC (window or through-wall)                                                NaN        2023$    350.160000    364.332000     311.640000           Capacity   
2                   Air Sealing                          <40% Reduction                                                NaN   2023$/sqft      0.480450      0.894236       1.333659  Leakage Reduction   
3                   Air Sealing                          >40% Reduction                                                NaN   2023$/sqft      

In [8]:
# Data Cleaning: Create unique row id and reorder colums
def create_row_id(component: str, class_name: str) -> str:
    """
    Create a clean, unique, Python-friendly identifier from component and class.
    
    Args:
        component: REMDB component name (e.g., "Air Source Heat Pump")
        class_name: REMDB class name (e.g., "Centrally ducted")
        
    Returns:
        Clean identifier string (e.g., "air_source_heat_pump_centrally_ducted")
        
    Examples:
        >>> create_row_id("Air Source Heat Pump", "Centrally ducted")
        'air_source_heat_pump_centrally_ducted'
        
        >>> create_row_id("Air Sealing", "<40% Reduction")
        'air_sealing_lt40pct_reduction'
    """
    # Handle missing class names
    if pd.isna(class_name):
        class_name = 'default'
    
    # Combine component and class
    raw_id = f"{component}_{class_name}"
    clean_id = raw_id.lower()
    
    # Handle comparison operators meaningfully
    clean_id = clean_id.replace('<40%', 'lt40pct')   # less than 40%
    clean_id = clean_id.replace('>40%', 'gt40pct')   # greater than 40%
    clean_id = clean_id.replace('<', 'lt')
    clean_id = clean_id.replace('>', 'gt')
    clean_id = clean_id.replace('%', 'pct')
    
    # Remove/replace problematic characters
    clean_id = re.sub(r'[()\[\]]', '', clean_id)     # Remove brackets
    clean_id = re.sub(r'[,/]', '_', clean_id)        # Replace , and / with _
    clean_id = re.sub(r'[\s\-]+', '_', clean_id)     # Replace spaces/hyphens with _
    clean_id = re.sub(r'_+', '_', clean_id)          # Collapse multiple underscores
    clean_id = clean_id.strip('_')                   # Remove leading/trailing underscores
    
    return clean_id

# Create a unique row id
df_remdb_tare_costs['row_id'] = df_remdb_tare_costs.apply(
    lambda row: create_row_id(row['component'].strip(), row['class']), 
    axis=1
)

print(df_remdb_tare_costs)

                      component                                   class                                  class_description output_units  pm1_coef_low  pm1_coef_mid  pm1_coef_high         pm1_metric  \
0               Air Conditioner                        Centrally ducted                                                NaN        2023$    196.966095    328.276825     459.587555   Cooling Capacity   
1               Air Conditioner        Room AC (window or through-wall)                                                NaN        2023$    350.160000    364.332000     311.640000           Capacity   
2                   Air Sealing                          <40% Reduction                                                NaN   2023$/sqft      0.480450      0.894236       1.333659  Leakage Reduction   
3                   Air Sealing                          >40% Reduction                                                NaN   2023$/sqft      0.107294      0.867979       0.785978  Leakage Reductio

In [9]:
# Select and reorder columns
FINAL_COLUMNS = [
    # Identification
    'row_id',
    'tare_category',
    'component',
    'class',
    'output_units',
    
    # Performance Metric 1 (primary)
    'pm1_metric',
    'pm1_unit',
    'pm1_coef_low',
    'pm1_coef_mid',
    'pm1_coef_high',
    'pm1_lower_bound',
    'pm1_upper_bound',
    
    # Performance Metric 2 (secondary, may be null)
    'pm2_metric',
    'pm2_unit',
    'pm2_coef_low',
    'pm2_coef_mid',
    'pm2_coef_high',
    'pm2_lower_bound',
    'pm2_upper_bound',
    
    # Intercepts
    'intercept_low',
    'intercept_mid',
    'intercept_high',
    
    # Installed cost adder and multiplier
    # 'install_method',
    'multiplier_retrofit',
    'adder_retrofit',
    
    # Metadata
    'lifetime_years',
    # 'data_quality_rank',
    # 'data_sources',
]

df_remdb_tare_clean = df_remdb_tare_costs[FINAL_COLUMNS].copy()

print(f"""
Final cleaned dataframe for retrofit capital costs:
{df_remdb_tare_clean}

Final columns ({len(FINAL_COLUMNS)}):
--------------------------------------------------""")
for i, col in enumerate(FINAL_COLUMNS, 1):
    print(f"  {i:2}. {col}")


Final cleaned dataframe for retrofit capital costs:
                                                row_id  tare_category                   component                                   class output_units         pm1_metric      pm1_unit  pm1_coef_low  \
0                     air_conditioner_centrally_ducted        cooling             Air Conditioner                        Centrally ducted        2023$   Cooling Capacity          Tons    196.966095   
1       air_conditioner_room_ac_window_or_through_wall        cooling             Air Conditioner        Room AC (window or through-wall)        2023$           Capacity          Tons    350.160000   
2                        air_sealing_lt40pct_reduction      enclosure                 Air Sealing                          <40% Reduction   2023$/sqft  Leakage Reduction       Percent      0.480450   
3                        air_sealing_gt40pct_reduction      enclosure                 Air Sealing                          >40% Reduction   202

In [10]:
REMBD_PROCESSED_PATH = os.path.join(PROJECT_ROOT, 'cmu_tare_model', 'data', 'retrofit_costs', 'remdb_v4_tare_retrofit_costs.csv')  # Processed REMDB v4 file path

df_remdb_tare_clean.to_csv(REMBD_PROCESSED_PATH)